# Generating text with a decoder, one token at a time

A small generative language model continues a sentence. The notebook opens
the model's loop and shows each step of the slides *Decoder Generative Model*
(after Jurafsky & Martin, *Speech and Language Processing*, 3rd ed.):

```
prompt ──1. tokenize──► token ids
         ──2. transformer layers──► one vector per token
                                    │ only the LAST position is used
         ──2. last linear layer (lm_head)──► logits u: one score per vocabulary token
         ──3. softmax(u / τ)──► probabilities y
         ──4. look at the top-10 candidates
         ──5. choose one token (temperature τ)
         ──6. append it to the input and go back to 2
```

The model is a **decoder**: it reads the tokens from left to right and is
trained for one task only — give the probability of the **next** token.

## 1. Preparation

**Choose a model.** One line of the next cell is active, the other is a
comment; to switch, swap the comment and run the notebook again
(*Run → Run All Cells*).

|   | model | parameters | transformer layers | trained on |
|---|---|---|---|---|
| **A** | `distilgpt2` | 82M | 6 | web pages (OpenWebText), learning to imitate GPT-2 |
| **B** | `roneneldan/TinyStories-33M` | 69M | 4 | TinyStories: short, simple stories for small children, written by GPT-3.5/4 |

*Is there anything smaller than GPT-2?* GPT-2 itself starts at 124M
parameters and 12 layers. `distilgpt2` is a *distilled* GPT-2: half the
layers, trained to reproduce GPT-2's predictions, and it is the smallest
model of that family that still writes general English. Smaller models
exist, but they pay for it: the TinyStories models (1M to 33M in their
names) only know the simple language of children's
stories, and test models such as `sshleifer/tiny-gpt2` (0.1M parameters,
vectors of size 2) produce noise. In all of these, a large part of the
parameters is the vocabulary table: 50,257 tokens × 768 numbers ≈ 39M.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

# Choose ONE model (see the table above): uncomment its line, comment the other.
MODEL_NAME = "distilgpt2"                     # A
# MODEL_NAME = "roneneldan/TinyStories-33M"   # B

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).eval()   # eval(): inference only, no training

print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")
print(f"  transformer layers (blocks): {len(model.transformer.h)}")
print(f"  vector size per token:       {model.config.hidden_size}")
print(f"  vocabulary size |V|:         {model.config.vocab_size:,} tokens")
print(f"  last linear layer (lm_head): {model.lm_head}")

distilgpt2: 82M parameters
  transformer layers (blocks): 6
  vector size per token:       768
  vocabulary size |V|:         50,257 tokens
  last linear layer (lm_head): Linear(in_features=768, out_features=50257, bias=False)


## 2. Step 1 — the input sentence, tokenized

The prompt is the example of the slides. Other prompts are in the comments
(the third one suits model B).

GPT-2's tokenizer splits text into pieces from a vocabulary of 50,257
tokens; `Ġ` marks a token that starts with a space. A decoder adds no
`[CLS]` or `[SEP]`: the ids are just the pieces of the text.

In [2]:
PROMPT = "So long and thanks for"                         # the example of the slides
# PROMPT = "The patient was admitted to the hospital with"
# PROMPT = "Once upon a time, there was a little"         # suits model B

inputs = tokenizer(PROMPT, return_tensors="pt")
ids = inputs["input_ids"]                                 # shape (1, tokens): a batch with one text

print("shape:", tuple(ids.shape))
pd.DataFrame({"token": tokenizer.convert_ids_to_tokens(ids[0]),
              "id": ids[0].tolist()})

shape: (1, 5)


,token,id
0,So,2396
1,Ġlong,890
2,Ġand,290
3,Ġthanks,5176
4,Ġfor,329


## 3. Step 2 — through the transformer layers to the last linear layer

The token ids go through the **transformer layers** (blocks). Each block
takes one vector per token and returns one vector per token, of the same
size; each vector can only look at its own token and the tokens **before**
it (causal attention).

To predict the next token, only the vector of the **last position** is
used: it is the only one that has seen the whole prompt. The **last linear
layer** (`lm_head`) multiplies it by a matrix of shape
(vocabulary × vector size) and returns one score per vocabulary token: the
**logits** $u$, of shape $[1 \times |V|]$.

In [3]:
with torch.no_grad():                                                   # no gradients: we are not training
    out = model.transformer(input_ids=ids, output_hidden_states=True)   # the transformer layers

for layer, h in enumerate(out.hidden_states):
    label = "input (token embeddings)" if layer == 0 else f"after block {layer}"
    print(f"{label:26} {tuple(h[0].shape)}   (tokens × vector size)")

last_vector = out.last_hidden_state[0, -1]          # the vector of the LAST position
with torch.no_grad():
    logits = model.lm_head(last_vector)             # the last linear layer → u

print(f"\nlast vector {tuple(last_vector.shape)}  ×  lm_head {tuple(model.lm_head.weight.shape)}"
      f"  →  logits u {tuple(logits.shape)}")
print(f"logits range from {logits.min():.1f} to {logits.max():.1f}")

# The usual one-line call, model(ids), does the same two steps:
with torch.no_grad():
    print("same as model(ids).logits[0, -1]:", torch.allclose(model(ids).logits[0, -1], logits, atol=1e-4))

input (token embeddings)   (5, 768)   (tokens × vector size)
after block 1              (5, 768)   (tokens × vector size)
after block 2              (5, 768)   (tokens × vector size)
after block 3              (5, 768)   (tokens × vector size)
after block 4              (5, 768)   (tokens × vector size)
after block 5              (5, 768)   (tokens × vector size)
after block 6              (5, 768)   (tokens × vector size)

last vector (768,)  ×  lm_head (50257, 768)  →  logits u (50257,)
logits range from -89.1 to -58.0
same as model(ids).logits[0, -1]: True


The slides follow four candidates: *all*, *the*, *your*, *that*. Their
logits for this prompt:

In [4]:
SLIDE_WORDS = [" all", " the", " your", " that"]           # with the leading space, as in the text
slide_ids = [tokenizer.encode(word)[0] for word in SLIDE_WORDS]

pd.DataFrame({"token": [repr(w) for w in SLIDE_WORDS],
              "id": slide_ids,
              "logit u": logits[slide_ids].double().numpy()}).round(2)

,token,id,logit u
0,' all',477,-59.55
1,' the',262,-59.26
2,' your',534,-58.84
3,' that',326,-61.20


## 4. Step 3 — softmax: from logits to probabilities

$$y = \text{softmax}(u) \qquad y_i = \frac{e^{u_i}}{\sum_{j=1}^{|V|} e^{u_j}}$$

Every logit becomes a number between 0 and 1, the order is kept, and the
whole vector sums to 1. Adding the same constant to all logits does not
change the result, so it does not matter that the logits of model A are
all negative.

In [5]:
probs = torch.softmax(logits, dim=-1)

print(f"probabilities: {tuple(probs.shape)}, sum = {probs.sum():.4f}")
pd.DataFrame({"token": [repr(w) for w in SLIDE_WORDS],
              "logit u": logits[slide_ids].double().numpy().round(2),
              "probability y": probs[slide_ids].double().numpy().round(4)})

probabilities: (50257,), sum = 1.0000


,token,logit u,probability y
0,' all',-59.55,0.0634
1,' the',-59.26,0.0846
2,' your',-58.84,0.1284
3,' that',-61.20,0.0122


## 5. Step 4 — the top-10 candidates

The ten tokens with the highest probability. Rank 1 is what **greedy
decoding** would always choose.

In [6]:
def bar(p, width=50):
    # A probability as a bar of `width` cells: █ = filled (1 cell = 2%), ░ = empty.
    filled = round(p * width)
    return "█" * filled + "░" * (width - filled)

def top_k_table(logits, probs, k=10, chosen=None):
    # The k most probable tokens; `chosen` (a token id) is marked with ◄.
    top = torch.topk(probs, k)
    table = pd.DataFrame({
        "token": [repr(tokenizer.decode(i)) for i in top.indices],
        "id": top.indices.tolist(),
        "logit u": logits[top.indices].double().numpy().round(2),
        "probability": top.values.double().numpy().round(4),
        "": [bar(p.item()) for p in top.values],
    }, index=range(1, k + 1))
    if chosen is not None:
        table["chosen"] = ["◄" if i == chosen else "" for i in table["id"]]
    return table

top10 = torch.topk(probs, 10).values.sum().item()
print(f"top-10 together: {top10:.3f}   the other {len(probs) - 10:,} tokens: {1 - top10:.3f}")
top_k_table(logits, probs)

top-10 together: 0.695   the other 50,247 tokens: 0.305


,token,id,logit u,probability,
1,' reading',3555,-57.99,0.3001,███████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
2,' your',534,-58.84,0.1284,██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
3,' the',262,-59.26,0.0846,████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
4,' all',477,-59.55,0.0634,███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
5,' taking',2263,-60.50,0.0245,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
6,' watching',4964,-60.52,0.0239,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
7,' listening',8680,-60.71,0.0198,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
8,' checking',10627,-60.80,0.0181,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
9,' supporting',6493,-60.83,0.0176,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...
10,' being',852,-61.03,0.0144,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...


## 6. Step 5 — temperature: which candidate is selected

Instead of always taking rank 1 (greedy), we **sample**: draw a token at
random, each with its probability. Before the softmax, the logits are
divided by the **temperature** $\tau$:

$$y = \text{softmax}(u / \tau)$$

* $\tau < 1$: high probabilities grow, low ones shrink — towards greedy;
* $\tau = 1$: the model's own distribution;
* $\tau > 1$: probabilities get closer to each other — towards uniform;
* $\tau = 0$ would divide by zero; it is taken as its limit: all the
  probability on rank 1, i.e. greedy decoding.

**Set the temperature** in the next cell; it is used in the rest of the
notebook. The table shows the same ten tokens at several temperatures, as
in the slides, and the probability left to the rest of the vocabulary.

In [7]:
TEMPERATURE = 0.7     # try 0 (greedy), 0.3, 1.0, 1.5 ...
SEED = 42             # same seed → same random draws on every run

def apply_temperature(logits, temperature):
    if temperature == 0:                            # limit τ → 0: greedy
        probs = torch.zeros_like(logits)
        probs[logits.argmax()] = 1.0
        return probs
    return torch.softmax(logits / temperature, dim=-1)

def choose_token(probs):
    return torch.multinomial(probs, num_samples=1).item()   # one random draw, weighted by probs

top_ids = torch.topk(logits, 10).indices
comparison = pd.DataFrame({f"τ={t}": apply_temperature(logits, t)[top_ids].double().numpy()
                           for t in [0.1, 0.5, 1, 2, 10]},
                          index=[repr(tokenizer.decode(i)) for i in top_ids])
comparison.loc["(rest of vocabulary)"] = (1 - comparison.sum()).clip(lower=0)
comparison.round(3)

,τ=0.1,τ=0.5,τ=1,τ=2,τ=10
' reading',1.0,0.739,0.300,0.022,0.000
' your',0.0,0.135,0.128,0.014,0.000
' the',0.0,0.059,0.085,0.011,0.000
' all',0.0,0.033,0.063,0.010,0.000
' taking',0.0,0.005,0.024,0.006,0.000
' watching',0.0,0.005,0.024,0.006,0.000
' listening',0.0,0.003,0.020,0.006,0.000
' checking',0.0,0.003,0.018,0.005,0.000
' supporting',0.0,0.003,0.018,0.005,0.000
' being',0.0,0.002,0.014,0.005,0.000


Twenty draws for the **same** prompt at the chosen temperature: the
model's output is the same each time, but the selected token changes.

In [8]:
torch.manual_seed(SEED)
probs_t = apply_temperature(logits, TEMPERATURE)
draws = [repr(tokenizer.decode(choose_token(probs_t))) for _ in range(20)]

print(f"τ = {TEMPERATURE}: 20 draws")
pd.Series(draws).value_counts().to_frame("times chosen")

τ = 0.7: 20 draws


,times chosen
' reading',13
' the',3
' your',2
' taking',1
' all',1


## 7. Step 6 — reinjecting the chosen token

**Autoregressive generation**: the chosen token is appended to the input,
and steps 2 to 5 run again on the longer sequence to choose the following
token, and so on until an end-of-text token (`<|endoftext|>`) or a length
limit.

### 7.1 One token at a time

The first cell starts from the prompt. **Run the second cell again and
again** (Ctrl+Enter) to add one token per run and see the top-10 from
which it was chosen (◄). Run the first cell to start over.

In [9]:
torch.manual_seed(SEED)
generated = tokenizer(PROMPT, return_tensors="pt")["input_ids"]     # step 1: start from the prompt
print(repr(tokenizer.decode(generated[0])))

'So long and thanks for'


In [10]:
# Run this cell again to add one more token.
with torch.no_grad():
    hidden = model.transformer(input_ids=generated).last_hidden_state    # step 2: transformer layers
    logits_step = model.lm_head(hidden[0, -1])                           #         + last linear layer
probs_step = apply_temperature(logits_step, TEMPERATURE)                 # steps 3 and 5: softmax(u / τ)
token_id = choose_token(probs_step)                                      # step 5: choose
generated = torch.cat([generated, torch.tensor([[token_id]])], dim=1)    # step 6: reinject

print(f"{generated.shape[1]} tokens:", repr(tokenizer.decode(generated[0])))
top_k_table(logits_step, probs_step, chosen=token_id)                    # step 4: the candidates

6 tokens: 'So long and thanks for taking'


,token,id,logit u,probability,,chosen
1,' reading',3555,-57.99,0.5415,███████████████████████████░░░░░░░░░░░░░░░░░░░...,
2,' your',534,-58.84,0.1611,████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,
3,' the',262,-59.26,0.0887,████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,
4,' all',477,-59.55,0.0587,███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,
5,' taking',2263,-60.50,0.0151,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,◄
6,' watching',4964,-60.52,0.0146,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,
7,' listening',8680,-60.71,0.0112,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,
8,' checking',10627,-60.80,0.0098,░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,
9,' supporting',6493,-60.83,0.0094,░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,
10,' being',852,-61.03,0.0070,░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░...,


### 7.2 The whole loop

The same steps in a loop, as in the slides' algorithm:

```
i ← 1;  wᵢ ~ p(w)
while wᵢ ≠ EOS:  i ← i + 1;  wᵢ ~ p(wᵢ | w<ᵢ)
```

Each line of the trace shows the token chosen at that step, its
probability (after the temperature) and the three best candidates.

In [11]:
N_TOKENS = 20        # length limit

def generate(prompt, temperature, n_tokens, show_steps=False):
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"]                # step 1
    for step in range(1, n_tokens + 1):
        with torch.no_grad():
            hidden = model.transformer(input_ids=ids).last_hidden_state      # step 2
            logits = model.lm_head(hidden[0, -1])
        probs = apply_temperature(logits, temperature)                       # step 3 (+ τ)
        token_id = choose_token(probs)                                       # step 5
        if show_steps:                                                       # step 4, short
            top3 = torch.topk(probs, 3)
            candidates = "   ".join(f"{tokenizer.decode(i)!r} {p:.2f}" for p, i in zip(top3.values, top3.indices))
            print(f"{step:2}. {tokenizer.decode(token_id)!r:14} p={probs[token_id]:.2f}   top-3: {candidates}")
        if token_id == tokenizer.eos_token_id:                               # end of text
            break
        ids = torch.cat([ids, torch.tensor([[token_id]])], dim=1)            # step 6
    return tokenizer.decode(ids[0])

torch.manual_seed(SEED)
text = generate(PROMPT, TEMPERATURE, N_TOKENS, show_steps=True)
print("\n" + text)

 1. ' taking'      p=0.02   top-3: ' reading' 0.54   ' your' 0.16   ' the' 0.09
 2. ' the'         p=0.84   top-3: ' the' 0.84   ' time' 0.04   ' this' 0.03
 3. ' time'        p=1.00   top-3: ' time' 1.00   ' opportunity' 0.00   ' plunge' 0.00


 4. ' to'          p=0.96   top-3: ' to' 0.96   ' and' 0.02   '.' 0.01


 5. ' create'      p=0.00   top-3: ' read' 0.42   ' write' 0.08   ' check' 0.08


 6. ' this'        p=0.60   top-3: ' this' 0.60   ' a' 0.12   ' the' 0.11


 7. ' site'        p=0.02   top-3: ' post' 0.14   ' blog' 0.09   ' article' 0.08
 8. '!'            p=0.22   top-3: '.' 0.55   '!' 0.22   ',' 0.09
 9. '\n'           p=0.69   top-3: '\n' 0.69   '<|endoftext|>' 0.10   ' I' 0.04


10. '\n'           p=0.95   top-3: '\n' 0.95   'I' 0.01   'If' 0.01


11. '\n'           p=0.68   top-3: '\n' 0.68   'I' 0.06   'If' 0.04


12. '\n'           p=0.27   top-3: '\n' 0.27   'I' 0.11   'If' 0.09
13. '-'            p=0.03   top-3: '\n' 0.14   'I' 0.10   'If' 0.09
14. '\n'           p=0.30   top-3: '\n' 0.30   ' The' 0.07   ' I' 0.03
15. '-'            p=0.81   top-3: '-' 0.81   'If' 0.02   'Thanks' 0.01


16. '\n'           p=0.97   top-3: '\n' 0.97   '<|endoftext|>' 0.03   '\n\n' 0.00
17. '-'            p=0.96   top-3: '-' 0.96   'Thanks' 0.00   'If' 0.00


18. '\n'           p=0.97   top-3: '\n' 0.97   '<|endoftext|>' 0.03   '\n\n' 0.00


19. '-'            p=0.98   top-3: '-' 0.98   'Thanks' 0.00   'If' 0.00
20. '\n'           p=0.97   top-3: '\n' 0.97   '<|endoftext|>' 0.02   ' -' 0.00

So long and thanks for taking the time to create this site!



-
-
-
-



Note that each step runs the model over the **whole** sequence again,
although only the last vector is used. Real systems keep the vectors already
computed for earlier positions (a *KV cache*) and process only the new
token; the result is the same.

## 8. Comparing temperatures

Each prompt generated with several temperatures, 40 tokens at most (it
stops earlier if `<|endoftext|>` is chosen). `repr` shows line breaks as
`\n`. Add your own prompts to the list.

In [12]:
COMPARE_PROMPTS = [PROMPT, "The patient was admitted to the hospital with"]

for prompt in COMPARE_PROMPTS:
    print("=" * 20, repr(prompt))
    for t in [0, 0.3, 0.7, 1.0, 1.5]:
        torch.manual_seed(SEED)
        print(f"τ = {t}" + (" (greedy)" if t == 0 else ""))
        print("   ", repr(generate(prompt, t, 40)))
    print()

==================== 'So long and thanks for'
τ = 0 (greedy)


    'So long and thanks for reading!\n\nAdvertisements'
τ = 0.3


    'So long and thanks for reading!\n\nAdvertisements'
τ = 0.7


    'So long and thanks for taking the time to create this site!\n\n\n\n-\n-\n-\n-\n-\n-\n-\n-\n-\n-\n-\n-\n-\n-\n'
τ = 1.0


    'So long and thanks for taking the time to create this site! Feel free to submit your project and then again, we would love to hear what you think! It could have possibly become a huge time consuming thing!\n\n'
τ = 1.5


    'So long and thanks for taking Daddy first step into clarifying dependencies more famously: Simple-Oato29 Inspective Publishing Setting For Growing YamatoKicks Takes Once It ACGS Reflect Congregation Sal Ruselve Dal made explicit Alexandria'

==================== 'The patient was admitted to the hospital with'
τ = 0 (greedy)


    'The patient was admitted to the hospital with a broken collarbone.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'
τ = 0.3


    'The patient was admitted to the hospital with a broken leg and was treated for a broken leg.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'
τ = 0.7


    'The patient was admitted to the hospital with a fractured ankle and is expected to recover more quickly.\n\n\n\n\nA spokeswoman for the NHS said: "The recent Government\'s response to an outbreak of Salmonella in the UK is'
τ = 1.0


    'The patient was admitted to the hospital with solid and non-fatal respiratory shock.'
τ = 1.5


    'The patient was admitted to the hospital with solid pleural surgical device at Intorexology Zurich, Vice La Rue Gladiolano Publishing, Taupt Zurich 2004)."Rev989 It AC 630 Thomas Colombat Sal Meielve Dalou Park Alexandria'



## What to notice (model A)

* **Five tokens in, 50,257 scores out.** The prompt becomes 5 tokens; each
  of the 6 blocks returns 5 vectors of 768 numbers; the last of them, times
  the `lm_head` matrix (50,257 × 768), gives one logit per vocabulary token.
  The one-line `model(ids)` gives exactly the same logits.
* **Logits are not probabilities.** Here they are all negative (−89.1 to
  −58.0); only their differences matter. After the softmax they are between
  0 and 1 and sum to 1.
* **The slides' four words are real candidates**, but in a different order
  and with smaller probabilities: *your* 0.128, *the* 0.085, *all* 0.063,
  *that* 0.012 (slides: *all* .44, *the* .33, *your* .15, *that* .08). The
  slides show only four tokens; here 50,257 share the probability, and the
  first place goes to *reading* (0.300), from the web pages the model was
  trained on.
* **The tail is large.** The top-10 together hold 0.695; the other 50,247
  tokens hold 0.305. Each of them is unlikely, but together they are drawn
  almost one time in three at τ = 1 (slide *Alas, random sampling doesn't
  work very well*).
* **Temperature moves that mass.** At τ = 0.5, *reading* goes up to 0.739
  and the rest of the vocabulary down to 0.014; at τ = 2 the rest holds
  0.910, at τ = 10 0.999 — nearly uniform over 50,257 tokens. In 20 draws
  at τ = 0.7, *reading* came 13 times, *the* 3, *your* 2, *taking* and
  *all* once each.
* **One unlikely choice changes everything after it.** The trace (τ = 0.7)
  starts with *taking*, a 0.02 choice; from there "the time to" is almost
  certain (0.84, 1.00, 0.96). At step 5, *create* is drawn from the tail
  (p < 0.005) instead of *read* (0.42). The top-10 table of 7.1 shows the
  probabilities after the temperature: *reading* has 0.54 there, against
  0.30 at τ = 1.
* **Repetition feeds itself.** After a few line breaks, the trace falls into
  `\n` `-` `\n` `-` …, each one chosen with p ≈ 0.97: every repetition makes
  the next one more probable.
* **Greedy is generic and repetitive** (section 8). For the slides' prompt,
  τ = 0 and τ = 0.3 give the same "reading!" and stop at `<|endoftext|>`
  after a typical web-page ending ("Advertisements"). For the hospital
  prompt, greedy writes "a broken collarbone." and then only line breaks;
  τ = 0.3 repeats itself ("a broken leg and was treated for a broken leg").
  τ = 0.7 reads like a news article; at τ = 1.0 the phrases start to be odd
  ("solid and non-fatal respiratory shock"), and at τ = 1.5 the text
  breaks down into invented words and names.

## Questions

1. In step 2, the transformer returns one vector per token, but only the
   last one is used. What would the vector of position 2 predict?
2. Why does greedy decoding ($\tau = 0$) give the same text on every run,
   and the other temperatures not? Change `SEED` and run section 8 again.
3. At which temperature does the text stop making sense? Compare the
   "rest of vocabulary" row of the table in section 6 with what you see.
4. Try the other prompts of step 1, and model B with the
   *Once upon a time* prompt. What does a model trained only on children's
   stories do with the hospital prompt?
5. The model never sees words, only token ids. Find, in a trace, a word
   that was generated in two or more pieces.